<a href="https://colab.research.google.com/github/RAJAMURUGAN-VS/genai-learning-journey/blob/main/04-rag/01_basic_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load Document

In [ ]:
!pip install -U langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from pprint import pprint

In [3]:
file_path = "https://arxiv.org/pdf/1706.03762"
loader = PyPDFLoader(file_path)
doc = loader.load()

Splitting Document

In [4]:
!pip install -qU langchain-text-splitters

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)

all_splits=text_splitter.split_documents(doc)

Creating Embeddings

In [7]:
! pip install -qU langchain langchain-huggingface sentence_transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(
  model_name="sentence-transformers/all-mpnet-base-v2"
)

In [ ]:
!pip install -U langchain-chroma

In [10]:
from langchain_chroma import Chroma

vector_store=Chroma(
  collection_name="research_collection",
  embedding_function=embedding_model,
  persist_directory="./chroma_langchain_db"
)
document_ids=vector_store.add_documents(documents=all_splits)

In [11]:
sample=vector_store.get(limit=1, include=["embeddings", "documents"])

Retrieve and Generate

In [12]:
def retrieve_context(query: str, k: int = 2):
  retrieved_docs=vector_store.similarity_search(query, k=k)

  docs_content=""
  for doc in retrieved_docs:
    docs_content+=f"Source: {doc.metadata}\n"
    docs_content+=f"Content: {doc.page_content}\n\n"

  return docs_content, retrieved_docs

Model

In [ ]:
!pip install -U langchain-google-genai

Initializing The Model

In [14]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key=userdata.get('GEMINI_API_KEY')
model=init_chat_model(
   "google_genai:gemini-2.5-flash",
   api_key=api_key,
)

Defining the Query function

In [19]:
def docu_chat(user_query):
  context, source_docs=retrieve_context(user_query, k=2)

  #Setting Up the LLM Instructions

  system_message=f"""You are a helpful chatbot.
                     Use only the following pieces of context to answer the
                     question. Don't makeup any new information: {context} """

  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_query}
  ]

  #Invoking LLM And Getting Results

  response=model.invoke(messages)
  return {
    "answer": response.content,
    "source_documents": source_docs,
    "context_used": context
  }

In [20]:
result=docu_chat( "Explain what is the use of decoders in transformers?")

In [23]:
pprint(result)

{'answer': 'In Transformers, decoders are part of the overall architecture '
           'that uses stacked self-attention and point-wise, fully connected '
           'layers.\n'
           '\n'
           'Specifically, the decoder has two main uses related to its '
           'attention mechanisms:\n'
           '*   **Encoder-decoder attention**: The decoder layers query from '
           'their previous layer and use memory keys and values from the '
           "encoder's output. This enables every position in the decoder to "
           'attend over all positions in the input sequence.\n'
           '*   **Decoder self-attention**: Self-attention layers in the '
           'decoder allow each position in the decoder to attend to all '
           'positions in the decoder up to and including that position.',
 'context_used': "Source: {'producer': 'pdfTeX-1.40.25', 'total_pages': 15, "
                 "'ptex.fullbanner': 'This is pdfTeX, Version "
                 '3.141592653-2.6-

In [22]:
print(result["answer"])

In Transformers, decoders are part of the overall architecture that uses stacked self-attention and point-wise, fully connected layers.

Specifically, the decoder has two main uses related to its attention mechanisms:
*   **Encoder-decoder attention**: The decoder layers query from their previous layer and use memory keys and values from the encoder's output. This enables every position in the decoder to attend over all positions in the input sequence.
*   **Decoder self-attention**: Self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position.
